## Building tiny transformer

Class for our tiny arithmetic transformer model.
N = 100
d_model = 20
multiheads = 4

It will have 3 layers for FFN, with Relu activation and a layernorm pre-activation for each layer.

This model will be a decoder only transformer that leverage "prefix language modeling" by allowing full attention over the input arithmetic expression and then causally generating the numerical result.

Since this project is for learning puposes also, I am building every part of transformer from scratch.

In [1]:
import torch
import torch.nn as nn

# An unoptimized vanilla MHA
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.q_linear_heads = nn.ModuleList([nn.Linear(d_model, self.d_k) for _ in range(num_heads)])
        self.k_linear_heads = nn.ModuleList([nn.Linear(d_model, self.d_k) for _ in range(num_heads)])
        self.v_linear_heads = nn.ModuleList([nn.Linear(d_model, self.d_k) for _ in range(num_heads)])

        self.out_linear = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        head_outputs = []
        
        for i in range(self.num_heads):
            q_i = self.q_linear_heads[i](x)
            k_i = self.k_linear_heads[i](x)
            v_i = self.v_linear_heads[i](x)
            
            scores_i = torch.matmul(q_i, k_i.transpose(-2, -1))
            scores_i = scores_i / (self.d_k ** 0.5)
            
            attention_weights_i = torch.softmax(scores_i, dim=-1)
            attention_weights_i = self.dropout(attention_weights_i)
            
            out_i = torch.matmul(attention_weights_i, v_i)
            head_outputs.append(out_i)
            
        concat_out = torch.cat(head_outputs, dim=-1)
        final_output = self.out_linear(concat_out)
        
        return final_output


In [2]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term)
        # apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)